# Phase 2 — LLM API Config Parameters

Focused exploration of generation config across **OpenAI**, **Anthropic**, and **Gemini**.

No tools. No chat history. Just the core config parameters that control how the model generates text.

---

## Parameters covered

| Parameter | What it controls |
|---|---|
| `temperature` | Randomness / creativity |
| `max_tokens` | Maximum output length |
| `top_p` | Nucleus sampling cutoff |
| `stop` | Stop sequences — halt generation at a string |

---

## 0. Setup

In [ ]:
# Install dependencies
%pip install openai anthropic google-genai python-dotenv --quiet

In [ ]:
# .env file should contain:
# OPENAI_API_KEY=your_key
# ANTHROPIC_API_KEY=your_key
# GEMINI_API_KEY=your_key

from dotenv import load_dotenv
load_dotenv()
print("Environment loaded.")

---

## Part 1 — OpenAI

Docs: https://platform.openai.com/docs/api-reference/responses


In [ ]:
from openai import OpenAI

client = OpenAI()
MODEL  = "gpt-4o-mini"

### 1.1 `temperature`

Controls randomness. Range: `0.0` – `2.0`

- `0.0` → always picks the highest-probability token (deterministic)
- `0.7` → balanced, good default
- `1.5+` → creative, unpredictable

Run the cell multiple times with different values and compare.

In [ ]:
SYSTEM_PROMPT  = "You are a helpful assistant."
USER_QUESTION  = "Give me one creative name for an AI startup."

for temp in [0.0, 0.7, 1.5]:
    response = client.responses.create(
        model=MODEL,
        instructions=SYSTEM_PROMPT,
        input=USER_QUESTION,
        temperature=temp
    )
    print(f"temperature={temp}: {response.output_text.strip()}")

### 1.2 `max_output_tokens`

Hard cap on output length. The model stops generating once this limit is hit — even mid-sentence.

- Does **not** affect quality, only length
- Useful for cost control and enforcing concise responses

In [ ]:
USER_QUESTION = "Explain how transformers work."

for max_tok in [20, 100, 500]:
    response = client.responses.create(
        model=MODEL,
        instructions=SYSTEM_PROMPT,
        input=USER_QUESTION,
        max_output_tokens=max_tok,
        temperature=0.7
    )
    text = response.output_text.strip()
    print(f"\n--- max_output_tokens={max_tok} ---")
    print(text)

### 1.3 `top_p` (nucleus sampling)

Cuts the vocabulary to the smallest set of tokens whose cumulative probability reaches `top_p`.

- `top_p=1.0` → consider all tokens (no cutoff)
- `top_p=0.9` → only the top tokens summing to 90% probability
- `top_p=0.1` → very narrow — only the most likely tokens

> **Tip:** Use either `temperature` or `top_p` to control randomness, not both at once.

In [ ]:
USER_QUESTION = "Continue this sentence: The future of AI is"

for top_p in [0.1, 0.9, 1.0]:
    response = client.responses.create(
        model=MODEL,
        instructions=SYSTEM_PROMPT,
        input=USER_QUESTION,
        top_p=top_p,
        temperature=1.0,          # keep temperature fixed, vary top_p
        max_output_tokens=60
    )
    print(f"\ntop_p={top_p}: {response.output_text.strip()}")

### 1.4 `stop` sequences

A list of strings that immediately halt generation when encountered.

- The stop string itself is **not** included in the output
- Useful for: structured output, preventing overgeneration, controlling format
- OpenAI accepts up to 4 stop sequences

In [ ]:
USER_QUESTION = "List 5 programming languages, one per line, numbered."

# Stop after the third item
response = client.responses.create(
    model=MODEL,
    instructions=SYSTEM_PROMPT,
    input=USER_QUESTION,
    stop=["4."],                  # halts when "4." is about to be written
    temperature=0.0,
    max_output_tokens=200
)

print("With stop=['4.']:")
print(response.output_text.strip())

### 1.5 All config together — OpenAI

In [ ]:
SYSTEM_PROMPT = "You are a concise technical assistant. Answer in plain English."
USER_QUESTION = "What is gradient descent?"

CONFIG = {
    "temperature": 0.3,
    "max_output_tokens": 150,
    "top_p": 0.9,
    "stop": ["In summary", "To conclude"]
}

response = client.responses.create(
    model=MODEL,
    instructions=SYSTEM_PROMPT,
    input=USER_QUESTION,
    **CONFIG
)

print(response.output_text.strip())

---

## Part 2 — Anthropic Claude

Docs: https://docs.anthropic.com/en/api/messages

> ⚠️ `max_tokens` is **required** for every Anthropic call — no default exists.

In [ ]:
import anthropic

client = anthropic.Anthropic()
MODEL  = "claude-sonnet-4-5"

### 2.1 `temperature`

Same concept as OpenAI. Range: `0.0` – `1.0` (Anthropic caps at 1.0, not 2.0).

In [ ]:
SYSTEM_PROMPT = "You are a helpful assistant."
USER_QUESTION = "Give me one creative name for an AI startup."

for temp in [0.0, 0.5, 1.0]:
    response = client.messages.create(
        model=MODEL,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": USER_QUESTION}],
        max_tokens=50,            # required
        temperature=temp
    )
    print(f"temperature={temp}: {response.content[0].text.strip()}")

### 2.2 `max_tokens`

Required in every call. Controls the hard output limit.

Claude's max context window varies by model:
- `claude-sonnet-4-5` → up to 8192 output tokens
- `claude-opus-4-5`   → up to 8192 output tokens

In [ ]:
USER_QUESTION = "Explain how transformers work."

for max_tok in [20, 100, 500]:
    response = client.messages.create(
        model=MODEL,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": USER_QUESTION}],
        max_tokens=max_tok,
        temperature=0.3
    )
    text = response.content[0].text.strip()
    print(f"\n--- max_tokens={max_tok} ---")
    print(text)
    print(f"Stop reason: {response.stop_reason}")

> **`stop_reason`** tells you *why* generation stopped:
> - `end_turn` → model finished naturally
> - `max_tokens` → hit the token limit (response was cut off)
> - `stop_sequence` → a stop string was hit
>
> Only Anthropic exposes this directly on the response object.

### 2.3 `top_p`

Same nucleus sampling as OpenAI. Range: `0.0` – `1.0`.

> Anthropic recommends using **either** `temperature` or `top_p`, not both.

In [ ]:
USER_QUESTION = "Continue this sentence: The future of AI is"

for top_p in [0.1, 0.9, 1.0]:
    response = client.messages.create(
        model=MODEL,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": USER_QUESTION}],
        max_tokens=60,
        top_p=top_p
        # no temperature when using top_p
    )
    print(f"\ntop_p={top_p}: {response.content[0].text.strip()}")

### 2.4 `stop_sequences`

Same concept as OpenAI's `stop`, but the parameter name is `stop_sequences` (plural, list).

In [ ]:
USER_QUESTION = "List 5 programming languages, one per line, numbered."

response = client.messages.create(
    model=MODEL,
    system=SYSTEM_PROMPT,
    messages=[{"role": "user", "content": USER_QUESTION}],
    max_tokens=200,
    temperature=0.0,
    stop_sequences=["4."]         # note: stop_sequences, not stop
)

print("With stop_sequences=['4.']:")
print(response.content[0].text.strip())
print(f"Stop reason: {response.stop_reason}")

### 2.5 All config together — Anthropic

In [ ]:
SYSTEM_PROMPT = "You are a concise technical assistant. Answer in plain English."
USER_QUESTION = "What is gradient descent?"

CONFIG = {
    "max_tokens": 150,
    "temperature": 0.3,
    "top_p": 0.9,
    "stop_sequences": ["In summary", "To conclude"]
}

response = client.messages.create(
    model=MODEL,
    system=SYSTEM_PROMPT,
    messages=[{"role": "user", "content": USER_QUESTION}],
    **CONFIG
)

print(response.content[0].text.strip())
print(f"\nStop reason: {response.stop_reason}")
print(f"Input tokens: {response.usage.input_tokens}")
print(f"Output tokens: {response.usage.output_tokens}")

> **`response.usage`** — Anthropic returns token counts on every response.
> Useful for tracking cost. OpenAI also returns usage; Gemini returns it via `response.usage_metadata`.

---

## Part 3 — Google Gemini

Docs: https://ai.google.dev/gemini-api/docs/text-generation

In [ ]:
from google import genai
from google.genai import types

client = genai.Client()
MODEL  = "gemini-2.5-flash-lite"

### 3.1 `temperature`

Same concept. Range: `0.0` – `2.0`.

Gemini config lives inside a `types.GenerateContentConfig` object — not a plain dict.

In [ ]:
SYSTEM_PROMPT = "You are a helpful assistant."
USER_QUESTION = "Give me one creative name for an AI startup."

for temp in [0.0, 0.7, 1.5]:
    CONFIG = types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT,
        temperature=temp
    )
    response = client.models.generate_content(
        model=MODEL,
        contents=USER_QUESTION,
        config=CONFIG
    )
    print(f"temperature={temp}: {response.text.strip()}")

### 3.2 `max_output_tokens`

Same concept as the other providers, same parameter name as OpenAI.

In [ ]:
USER_QUESTION = "Explain how transformers work."

for max_tok in [20, 100, 500]:
    CONFIG = types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT,
        temperature=0.3,
        max_output_tokens=max_tok
    )
    response = client.models.generate_content(
        model=MODEL,
        contents=USER_QUESTION,
        config=CONFIG
    )
    print(f"\n--- max_output_tokens={max_tok} ---")
    print(response.text.strip())
    print(f"Finish reason: {response.candidates[0].finish_reason}")

### 3.3 `top_p` and `top_k`

Gemini supports both nucleus sampling (`top_p`) and top-k sampling (`top_k`) — unique among the three providers.

- `top_k` → only consider the top K most probable tokens at each step
- `top_p` → only consider tokens summing to probability P

Gemini applies `top_k` first, then `top_p` on the filtered set.

In [ ]:
USER_QUESTION = "Continue this sentence: The future of AI is"

# top_p only
for top_p in [0.1, 0.9]:
    CONFIG = types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT,
        temperature=1.0,
        top_p=top_p,
        max_output_tokens=60
    )
    response = client.models.generate_content(
        model=MODEL,
        contents=USER_QUESTION,
        config=CONFIG
    )
    print(f"top_p={top_p}: {response.text.strip()}")

print()

# top_k (Gemini only)
for top_k in [1, 10, 40]:
    CONFIG = types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT,
        temperature=1.0,
        top_k=top_k,
        max_output_tokens=60
    )
    response = client.models.generate_content(
        model=MODEL,
        contents=USER_QUESTION,
        config=CONFIG
    )
    print(f"top_k={top_k}: {response.text.strip()}")

### 3.4 `stop_sequences`

Same as Anthropic — the parameter is called `stop_sequences` (list of strings).

In [ ]:
USER_QUESTION = "List 5 programming languages, one per line, numbered."

CONFIG = types.GenerateContentConfig(
    system_instruction=SYSTEM_PROMPT,
    temperature=0.0,
    max_output_tokens=200,
    stop_sequences=["4."]         # same name as Anthropic
)

response = client.models.generate_content(
    model=MODEL,
    contents=USER_QUESTION,
    config=CONFIG
)

print("With stop_sequences=['4.']:")
print(response.text.strip())
print(f"Finish reason: {response.candidates[0].finish_reason}")

### 3.5 All config together — Gemini

In [ ]:
SYSTEM_PROMPT = "You are a concise technical assistant. Answer in plain English."
USER_QUESTION = "What is gradient descent?"

CONFIG = types.GenerateContentConfig(
    system_instruction=SYSTEM_PROMPT,
    temperature=0.3,
    max_output_tokens=150,
    top_p=0.9,
    top_k=40,
    stop_sequences=["In summary", "To conclude"]
)

response = client.models.generate_content(
    model=MODEL,
    contents=USER_QUESTION,
    config=CONFIG
)

print(response.text.strip())
print(f"\nFinish reason: {response.candidates[0].finish_reason}")
print(f"Input tokens:  {response.usage_metadata.prompt_token_count}")
print(f"Output tokens: {response.usage_metadata.candidates_token_count}")

---

## Side-by-Side Config Reference

| Parameter | OpenAI | Anthropic | Gemini |
|---|---|---|---|
| Randomness | `temperature` (0–2) | `temperature` (0–1) | `temperature` (0–2) |
| Max output | `max_output_tokens` | `max_tokens` (**required**) | `max_output_tokens` |
| Nucleus sampling | `top_p` | `top_p` | `top_p` |
| Top-k sampling | ❌ not exposed | ❌ not exposed | ✅ `top_k` |
| Stop strings | `stop` (list, max 4) | `stop_sequences` (list) | `stop_sequences` (list) |
| Config style | plain `dict` | plain `dict` | `GenerateContentConfig()` |
| Stop reason | `response.status` | `response.stop_reason` | `candidates[0].finish_reason` |
| Token usage | `response.usage` | `response.usage` | `response.usage_metadata` |

---

## Key Takeaways

1. **`temperature` vs `top_p`** — use one, not both
2. **`max_tokens` in Anthropic is required** — always include it
3. **`stop` vs `stop_sequences`** — OpenAI uses `stop`, the other two use `stop_sequences`
4. **`top_k` is Gemini-only** — useful for tighter control than `top_p` alone
5. **Always check `stop_reason`/`finish_reason`** — tells you if the response was cut off (`max_tokens`) or finished naturally (`end_turn` / `STOP`)